# Content Embeddings — Weighted Tag Vectors & Cross-Domain Recommendations

Builds a content-based similarity signal from AniList genre/tag data, used for two purposes:
- **Cold-start handling** — recommending anime/manga with little or no rating history, based on what they're about rather than who liked them
- **Cross-domain bridging** — recommending manga based on an anime a user liked (and vice versa), without needing shared user identities across the two ratings datasets (which don't exist across platforms — see `01_data_collection.ipynb`)

**Approach note:** an earlier version of this step used sentence-transformer embeddings on plot synopses (and later, synopsis + genre/tag text combined) to measure similarity. That approach under-performed on real test cases — e.g. it rated *Mob Psycho 100* as more similar to *Attack on Titan* than *Fullmetal Alchemist: Brotherhood* was, which doesn't match genre-savvy fan intuition. The issue: raw synopsis text captures plot events, not tone/theme, and even tag-enriched text over-weighted common, low-signal tags ("Male Protagonist", "Shounen") while diluting rare, meaningful ones ("Cannibalism", "Steampunk") on titles with long tag lists. The approach below instead builds explicit **rarity-weighted tag vectors** (an IDF-style scheme), which directly fixes both problems — validated below against known-similar and known-different anime pairs.

In [4]:
import json
import os
import math
from collections import Counter

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz

## Load anime & manga metadata

Both datasets come from `01_data_collection.ipynb`'s AniList pull — same tag vocabulary, which is what makes cross-domain comparison valid later.

In [5]:
DATA_DIR = os.path.join("..", "data")

with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

with open(os.path.join(DATA_DIR, "manga_data.jsonl"), "r") as f:
    manga_content = [json.loads(line) for line in f]

print(len(anime_content), len(manga_content))

5000 5000


## Build rarity-weighted tag vectors

Each anime/manga is represented as a vector over the *combined* anime+manga tag vocabulary, where each tag's weight is scaled by how rare it is (an IDF-style scheme: `log(total_items / (times_this_tag_appears + 1))`).

**Why rarity weighting, not raw tag presence:** a tag like "Male Protagonist" appears on a huge fraction of all anime and tells you almost nothing when shared between two titles. A tag like "Cannibalism" or "Steampunk" is rare and genuinely indicates deep similarity when shared. Raw (unweighted) tag overlap treats both the same, and also gets diluted on titles with unusually long tag lists (more total tags to add unrelated noise). Rarity weighting fixes both issues at once.

**Why the vocabulary/weights are computed across anime + manga together, not separately:** for cross-domain similarity to mean anything, "Tragedy" needs to carry the same weight whether it's an anime's tag or a manga's tag — computing weights separately per domain would make the two vector spaces incomparable.

In [6]:
tag_counts = Counter()
for item in anime_content + manga_content:
    for t in item.get('tags', []):
        tag_counts[t['name']] += 1

total_items = len(anime_content) + len(manga_content)

def tag_weight(tag_name):
    # Rare tags get a high weight, common tags get pushed toward ~0
    return math.log(total_items / (tag_counts[tag_name] + 1))

all_tags = sorted(tag_counts.keys())
tag_to_col = {t: i for i, t in enumerate(all_tags)}

def build_tag_vector(item):
    vec = np.zeros(len(all_tags))
    for t in item.get('tags', []):
        vec[tag_to_col[t['name']]] = tag_weight(t['name'])
    return vec

anime_tag_vectors = np.array([build_tag_vector(a) for a in anime_content])
manga_tag_vectors = np.array([build_tag_vector(m) for m in manga_content])

print(anime_tag_vectors.shape, manga_tag_vectors.shape)

np.save(os.path.join(DATA_DIR, "anime_tag_vectors.npy"), anime_tag_vectors)
np.save(os.path.join(DATA_DIR, "manga_tag_vectors.npy"), manga_tag_vectors)

(5000, 422) (5000, 422)


## Cross-domain recommendation: anime → manga

Given an anime, finds the most similar manga by tag-vector cosine similarity — the actual cross-domain bridge described in the intro. Same-franchise matches (e.g. an anime's own source manga, or its direct spin-offs) are filtered out via fuzzy title matching, since the goal is genuine discovery, not surfacing the obvious "here's the manga this anime is based on" result at the top.

In [9]:
def is_same_franchise(title_a, title_b, threshold=60):
    romaji_score = fuzz.partial_ratio(title_a['romaji'] or '', title_b['romaji'] or '')
    english_score = fuzz.partial_ratio(title_a.get('english') or '', title_b.get('english') or '')
    return max(romaji_score, english_score) >= threshold

In [16]:
def recommend_manga_for_anime(anime_idx, k=10, exclude_same_franchise=True):
    anime_vec = anime_tag_vectors[anime_idx].reshape(1, -1)
    similarities = cosine_similarity(anime_vec, manga_tag_vectors)[0]

    source_title = anime_content[anime_idx]['title']

    # Pull more candidates than needed, since franchise matches get filtered out
    candidate_indices = similarities.argsort()[::-1][:k * 5]

    results = []
    for i in candidate_indices:
        candidate_title = manga_content[i]['title']
        if exclude_same_franchise and is_same_franchise(source_title, candidate_title):
            continue
        title = candidate_title['english'] or candidate_title['romaji']
        results.append((title, similarities[i]))
        if len(results) == k:
            break

    return results

In [19]:
for i in range(100):
    print(f'{i}: {anime_content[i]['title']['english']}')

0: Attack on Titan
1: Demon Slayer: Kimetsu no Yaiba
2: JUJUTSU KAISEN
3: Death Note
4: My Hero Academia
5: Hunter x Hunter (2011)
6: One-Punch Man
7: ONE PIECE
8: Tokyo Ghoul
9: Attack on Titan Season 2
10: Fullmetal Alchemist: Brotherhood
11: Naruto
12: Sword Art Online
13: A Silent Voice
14: Your Name.
15: Attack on Titan Season 3
16: My Hero Academia Season 2
17: Attack on Titan Final Season
18: The Promised Neverland
19: Assassination Classroom
20: Chainsaw Man
21: Mob Psycho 100
22: Re:ZERO -Starting Life in Another World-
23: Your lie in April
24: Attack on Titan Season 3 Part 2
25: Naruto: Shippuden
26: ERASED
27: My Hero Academia Season 3
28: Steins;Gate
29: HAIKYU!!
30: Black Clover
31: Rascal Does Not Dream of Bunny Girl Senpai
32: SPY x FAMILY
33: Kaguya-sama: Love is War
34: Dr. STONE
35: Violet Evergarden
36: No Game, No Life
37: Vinland Saga
38: Toradora!
39: Noragami
40: Akame ga Kill!
41: Horimiya
42: My Hero Academia Season 4
43: The Seven Deadly Sins
44: KONOSUBA -Go

In [17]:
print(anime_content[0]['title']['english'])
recommend_manga_for_anime(0)

Attack on Titan


[('Claymore', np.float64(0.48168429058014345)),
 ('Distant Sky', np.float64(0.43378065106477487)),
 ('Creature!', np.float64(0.3994437437843398)),
 ('Mosquito Wars', np.float64(0.39655871977733376)),
 ('The Horizon', np.float64(0.36947541669643746)),
 ('Kaiju No. 8: B-Side', np.float64(0.3631810165510793)),
 ('Talentless Nana', np.float64(0.36179019048708433)),
 ('Duty After School', np.float64(0.35986522400436416)),
 ('Kaiju No.8', np.float64(0.35203626818922784)),
 ('Fire Punch', np.float64(0.3490130303007308))]